In [1]:
import openml, warnings, numpy as np, pandas as pd
from tqdm.auto import tqdm
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import make_pipeline, Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import LogisticRegression, Ridge, Lasso, LinearRegression
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.ensemble import (RandomForestClassifier, RandomForestRegressor,
    ExtraTreesClassifier, ExtraTreesRegressor,
    HistGradientBoostingClassifier, HistGradientBoostingRegressor)
from sklearn.naive_bayes import GaussianNB
warnings.filterwarnings("ignore")

MAX_ROWS, CV = 5000, 5

CLF_MODELS = {
    "LogisticRegression": LogisticRegression(max_iter=500),
    "KNN": KNeighborsClassifier(),
    "DecisionTree": DecisionTreeClassifier(random_state=0),
    "RandomForest": RandomForestClassifier(n_estimators=100, n_jobs=-1, random_state=0),
    "ExtraTrees": ExtraTreesClassifier(n_estimators=100, n_jobs=-1, random_state=0),
    "HistGB": HistGradientBoostingClassifier(random_state=0),
    "NaiveBayes": GaussianNB(),
}
REG_MODELS = {
    "Ridge": Ridge(),
    "Lasso": Lasso(max_iter=5000),
    "KNN": KNeighborsRegressor(),
    "DecisionTree": DecisionTreeRegressor(random_state=0),
    "RandomForest": RandomForestRegressor(n_estimators=100, n_jobs=-1, random_state=0),
    "ExtraTrees": ExtraTreesRegressor(n_estimators=100, n_jobs=-1, random_state=0),
    "HistGB": HistGradientBoostingRegressor(random_state=0),
}

/Users/lucifer/Library/Python/3.9/lib/python/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def make_preprocessor(X):
    num = X.select_dtypes("number").columns
    cat = X.columns.difference(num)
    return ColumnTransformer([
        ("num", make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), num),
        ("cat", make_pipeline(SimpleImputer(strategy="most_frequent"),
                OneHotEncoder(handle_unknown="ignore", max_categories=20, sparse_output=False)), cat),
    ])

def landmarkers(X, y, task):
    lm = {"clf": {"lm_stump": DecisionTreeClassifier(max_depth=1),
                  "lm_1nn": KNeighborsClassifier(n_neighbors=1)},
          "reg": {"lm_stump": DecisionTreeRegressor(max_depth=1),
                  "lm_linear": LinearRegression()}}[task]
    scoring = "balanced_accuracy" if task == "clf" else "r2"
    return {k: cross_val_score(Pipeline([("p", make_preprocessor(X)), ("m", m)]),
                               X, y, cv=3, scoring=scoring).mean() for k, m in lm.items()}

def extract_meta_features(X, y, task):
    num = X.select_dtypes("number")
    n, d = X.shape
    f = {"n_samples": n, "log_n_samples": np.log(n), "n_features": d,
         "n_num": num.shape[1], "pct_cat": (d - num.shape[1]) / d,
         "pct_missing": X.isna().mean().mean(), "dim_ratio": d / n}
    if num.shape[1]:
        f["mean_skew"], f["mean_kurt"] = num.skew().mean(), num.kurt().mean()
        c = num.corr().abs().values
        f["mean_abs_corr"] = c[np.triu_indices_from(c, 1)].mean() if num.shape[1] > 1 else 0
    if task == "clf":
        p = y.value_counts(normalize=True).values
        f.update(n_classes=len(p), class_entropy=-(p * np.log(p)).sum(),
                 imbalance_ratio=p.max() / p.min())
    else:
        f.update(target_skew=y.skew(), target_kurt=y.kurt(), target_cv=y.std() / (abs(y.mean()) + 1e-9))
    f.update(landmarkers(X, y, task))
    return f

In [3]:
def process_task(task_id, task):
    t = openml.tasks.get_task(task_id)
    ds = t.get_dataset()
    X, y = t.get_X_and_y(dataset_format="dataframe")
    if len(X) > MAX_ROWS:
        idx = X.sample(MAX_ROWS, random_state=0).index
        X, y = X.loc[idx].reset_index(drop=True), y.loc[idx].reset_index(drop=True)
    models, scoring = (CLF_MODELS, "balanced_accuracy") if task == "clf" else (REG_MODELS, "r2")

    row = {"dataset": ds.name, "task_id": task_id, **extract_meta_features(X, y, task)}
    for name, m in models.items():
        pipe = Pipeline([("p", make_preprocessor(X)), ("m", m)])
        try:
            row[f"score_{name}"] = cross_val_score(pipe, X, y, cv=CV, scoring=scoring).mean()
        except Exception:
            row[f"score_{name}"] = np.nan
    return row

In [4]:
clf_tasks = openml.study.get_suite(99).tasks    # OpenML-CC18 (classification)
reg_tasks = openml.study.get_suite(269).tasks   # AutoML benchmark (regression)

def build(task_ids, task):
    rows = []
    for tid in tqdm(task_ids):
        try: rows.append(process_task(tid, task))
        except Exception as e: print("skip", tid, e)
    df = pd.DataFrame(rows)
    score_cols = [c for c in df if c.startswith("score_")]
    df = df.dropna(subset=score_cols)
    df["best_model"] = df[score_cols].idxmax(axis=1).str.replace("score_", "")
    return df

clf_meta = build(clf_tasks, "clf"); clf_meta.to_csv("../data/meta_clf.csv", index=False)
reg_meta = build(reg_tasks, "reg"); reg_meta.to_csv("../data/meta_reg.csv", index=False)
print(clf_meta.best_model.value_counts(), reg_meta.best_model.value_counts(), sep="\n\n")

 94%|█████████▍| 31/33 [32:32<00:25, 12.53s/it]  

skip 360932 cannot perform skew with type Sparse[float64, 0]


 97%|█████████▋| 32/33 [32:40<00:11, 11.18s/it]

skip 360933 cannot perform skew with type Sparse[float64, 0]


100%|██████████| 33/33 [32:56<00:00, 59.90s/it]

best_model
HistGB                21
LogisticRegression    17
ExtraTrees            13
NaiveBayes             8
RandomForest           5
KNN                    5
DecisionTree           3
Name: count, dtype: int64

best_model
HistGB          13
ExtraTrees       7
Lasso            5
Ridge            3
RandomForest     3
Name: count, dtype: int64
